In [34]:
import pandas as pd
import pyspark
from pyspark.sql import SparkSession

from pyspark.sql import types



In [2]:
spark = SparkSession.builder \
    .master("local[*]") \
    .appName('test') \
    .getOrCreate()

25/03/02 12:42:18 WARN Utils: Your hostname, DESKTOP-NSHTE4G resolves to a loopback address: 127.0.1.1; using 172.31.117.203 instead (on interface eth0)
25/03/02 12:42:18 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/03/02 12:42:23 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
25/03/02 12:42:26 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
25/03/02 12:42:43 WARN GarbageCollectionMetrics: To enable non-built-in garbage collector(s) List(G1 Concurrent GC), users should configure it(them) to spark.eventLog.gcMetrics.youngGenerationGarbageCollectors or spark.eventLog.gcMetrics.oldGenerationGarbageCollectors


In [ ]:
spark --version

In [4]:
!wget "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-10.parquet"

--2025-03-02 12:45:19--  https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-10.parquet
Resolving d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)... 18.239.238.133, 18.239.238.119, 18.239.238.152, ...
Connecting to d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)|18.239.238.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 64346071 (61M) [binary/octet-stream]
Saving to: ‘yellow_tripdata_2024-10.parquet’

yellow_tripdata_202  17%[==>                 ]  10.98M   342KB/s    in 8m 15s  

2025-03-02 12:53:47 (22.7 KB/s) - Connection closed at byte 11515640. Retrying.

--2025-03-02 12:53:48--  (try: 2)  https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-10.parquet
Connecting to d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)|18.239.238.133|:443... connected.
HTTP request sent, awaiting response... 206 Partial Content
Length: 64346071 (61M), 52830431 (50M) remaining [binary/octet-stream]
Sa

In [5]:
df = spark.read \
    .option("header", "true") \
    .parquet('yellow_tripdata_2024-10.parquet')

In [6]:
df.schema

StructType([StructField('VendorID', IntegerType(), True), StructField('tpep_pickup_datetime', TimestampNTZType(), True), StructField('tpep_dropoff_datetime', TimestampNTZType(), True), StructField('passenger_count', LongType(), True), StructField('trip_distance', DoubleType(), True), StructField('RatecodeID', LongType(), True), StructField('store_and_fwd_flag', StringType(), True), StructField('PULocationID', IntegerType(), True), StructField('DOLocationID', IntegerType(), True), StructField('payment_type', LongType(), True), StructField('fare_amount', DoubleType(), True), StructField('extra', DoubleType(), True), StructField('mta_tax', DoubleType(), True), StructField('tip_amount', DoubleType(), True), StructField('tolls_amount', DoubleType(), True), StructField('improvement_surcharge', DoubleType(), True), StructField('total_amount', DoubleType(), True), StructField('congestion_surcharge', DoubleType(), True), StructField('Airport_fee', DoubleType(), True)])

In [16]:
#!head -n 1001 yellow_tripdata_2024-10.parquet > head.parquet
df_pandas = pd.read_parquet('yellow_tripdata_2024-10.parquet', engine='pyarrow', columns=None)[:1000]


In [17]:
df_pandas.dtypes

VendorID                          int32
tpep_pickup_datetime     datetime64[us]
tpep_dropoff_datetime    datetime64[us]
passenger_count                 float64
trip_distance                   float64
RatecodeID                      float64
store_and_fwd_flag               object
PULocationID                      int32
DOLocationID                      int32
payment_type                      int64
fare_amount                     float64
extra                           float64
mta_tax                         float64
tip_amount                      float64
tolls_amount                    float64
improvement_surcharge           float64
total_amount                    float64
congestion_surcharge            float64
Airport_fee                     float64
dtype: object

In [18]:
spark.createDataFrame(df_pandas).schema

StructType([StructField('VendorID', LongType(), True), StructField('tpep_pickup_datetime', TimestampType(), True), StructField('tpep_dropoff_datetime', TimestampType(), True), StructField('passenger_count', DoubleType(), True), StructField('trip_distance', DoubleType(), True), StructField('RatecodeID', DoubleType(), True), StructField('store_and_fwd_flag', StringType(), True), StructField('PULocationID', LongType(), True), StructField('DOLocationID', LongType(), True), StructField('payment_type', LongType(), True), StructField('fare_amount', DoubleType(), True), StructField('extra', DoubleType(), True), StructField('mta_tax', DoubleType(), True), StructField('tip_amount', DoubleType(), True), StructField('tolls_amount', DoubleType(), True), StructField('improvement_surcharge', DoubleType(), True), StructField('total_amount', DoubleType(), True), StructField('congestion_surcharge', DoubleType(), True), StructField('Airport_fee', DoubleType(), True)])

In [49]:
yellow_schema = types.StructType([
    types.StructField("VendorID", types.IntegerType(), True),
    types.StructField("tpep_pickup_datetime", types.TimestampType(), True),
    types.StructField("tpep_dropoff_datetime", types.TimestampType(), True),
    types.StructField("passenger_count", types.LongType(), True),
    types.StructField("trip_distance", types.DoubleType(), True),
    types.StructField("RatecodeID", types.LongType(), True),
    types.StructField("store_and_fwd_flag", types.StringType(), True),
    types.StructField("PULocationID", types.IntegerType(), True),
    types.StructField("DOLocationID", types.IntegerType(), True),
    types.StructField("payment_type", types.LongType(), True),
    types.StructField("fare_amount", types.DoubleType(), True),
    types.StructField("extra", types.DoubleType(), True),
    types.StructField("mta_tax", types.DoubleType(), True),
    types.StructField("tip_amount", types.DoubleType(), True),
    types.StructField("tolls_amount", types.DoubleType(), True),
    types.StructField("improvement_surcharge", types.DoubleType(), True),
    types.StructField("total_amount", types.DoubleType(), True),
    types.StructField("congestion_surcharge", types.DoubleType(), True),
    types.StructField("Airport_fee", types.DoubleType(), True)
])

input_path = 'yellow_tripdata_2024-10.parquet'



df_yellow = spark.read \
    .option("header", "true") \
    .schema(yellow_schema) \
    .parquet(input_path)

df_yellow = df_yellow.repartition(4)
output_path = f'data/pq/yellow/2024/10/'


In [50]:
df_yellow.write.parquet(output_path)


In [51]:
ls

 03_test.ipynb                               09_spark_gcs.ipynb
 03_test.ipynb:Zone.Identifier               09_spark_gcs.ipynb:Zone.Identifier
 04_pyspark.ipynb                           'Module 5 - Batch Process HW.ipynb'
 04_pyspark.ipynb:Zone.Identifier            cloud.md
 05_taxi_schema.ipynb                        cloud.md:Zone.Identifier
 05_taxi_schema.ipynb:Zone.Identifier        data/
 06_spark_sql.ipynb                          download_data.sh
 06_spark_sql.ipynb:Zone.Identifier          download_data.sh:Zone.Identifier
 06_spark_sql.py                             head.csv
 06_spark_sql.py:Zone.Identifier             head.parquet
 06_spark_sql_big_query.py                   homework.ipynb
 06_spark_sql_big_query.py:Zone.Identifier   homework.ipynb:Zone.Identifier
 07_groupby_join.ipynb                       taxi_zone_lookup.csv
 07_groupby_join.ipynb:Zone.Identifier       yellow_tripdata_2024-10.parquet
 08_rdds.ipynb                               zones/
 08_rdds.ipynb:Zon